In [7]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [8]:
# Load Data
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()
print("Data loaded successfully!")

Data loaded successfully!


In [14]:
# XGBoost scale_pos_weight logic mapping
scale_pos_w_ratio = np.sum(y_train == 0) / np.sum(y_train == 1)

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', random_state=42),
    'XGBoost': XGBClassifier(scale_pos_weight=scale_pos_w_ratio, eval_metric='logloss', random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f"{name}: Mean ROC-AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")


Logistic Regression: Mean ROC-AUC = 0.8481 (+/- 0.0116)
Random Forest: Mean ROC-AUC = 0.8275 (+/- 0.0113)
XGBoost: Mean ROC-AUC = 0.8211 (+/- 0.0092)


In [16]:
y_proba = best_model.predict_proba(X_test)[:, 1]
print(f"Test Set ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

# Fixed grid parameters with standard array configurations
param_grid = {
    'max_depth':[3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators':[100, 200],
    'subsample': [0.8, 1.0]
}

# Base model instance creation using dynamic balance scaling ratio
xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos_w_ratio, 
    eval_metric='logloss', 
    random_state=42
)

# GridSearchCV Setup with full stratification rules
grid_search = GridSearchCV(
    estimator=xgb_base, 
    param_grid=param_grid, 
    cv=cv, 
    scoring='roc_auc', 
    n_jobs=-1, 
    verbose=1
)

print("Fitting GridSearchCV across the parameters matrix...")
grid_search.fit(X_train, y_train)

# Fetching the locked best model configuration
best_model = grid_search.best_estimator_
print(f"\nBest Hyperparameters Found: {grid_search.best_params_}")
print(f"Best CV ROC-AUC Score: {grid_search.best_score_:.4f}")


Test Set ROC-AUC Score: 0.8455
Fitting GridSearchCV across the parameters matrix...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best Hyperparameters Found: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Best CV ROC-AUC Score: 0.8485


In [17]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

# Absolute directory validation strategy
os.makedirs(os.path.abspath('../artifacts/models'), exist_ok=True)
model_path = os.path.abspath('../artifacts/models/best_model.joblib')

joblib.dump(best_model, model_path)
print(f"🎉 Model saved at: {model_path}")


              precision    recall  f1-score   support

           0       0.91      0.72      0.81      1035
           1       0.51      0.81      0.63       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.81      0.74      0.76      1409

🎉 Model saved at: c:\Users\Administrator\customer-churn-intelligence-platform\artifacts\models\best_model.joblib
